In [1]:
import pandas as pd
import numpy as np
import unicodedata
import re
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score
)
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

In [2]:
FILE = Path("df_final.csv")
data = pd.read_csv(FILE)

data

,Цена,Дата публикации,Город,is_class_eco,is_class_comfort,is_class_business,is_class_elite,is_brick,is_monolith,is_panel,...,Month_Public,DayOfWeek_Public,Floor_Ratio,Is_First_Floor,Is_Last_Floor,Area_per_Room,Infrastructure_Score,Площадь_log,Цена_log,Цена_за_квадратный_метр_log
0,5964400,2025-09-22 13:37:50,Киров,0,0,0,0,0,0,0,...,9,0,1.000000,0,1,24.000000,3.42,3.891820,15.601319,11.730126
1,5829810,2025-09-29 20:46:23,Киров,0,0,0,0,0,0,0,...,9,0,0.916667,0,0,39.000000,1.26,3.688879,15.578495,11.914940
2,6400900,2025-09-29 10:26:27,Киров,0,1,0,0,0,0,0,...,9,0,1.000000,0,1,17.333333,4.01,3.970292,15.671949,11.720714
3,5814900,2025-09-29 20:37:16,Киров,0,0,0,0,0,0,0,...,9,0,0.833333,0,0,39.000000,1.26,3.688879,15.575934,11.912379
4,6315750,2025-09-29 20:42:24,Киров,0,0,0,0,0,0,0,...,9,0,1.000000,0,1,40.000000,1.26,3.713572,15.658557,11.969684
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17504,6336000,2025-10-16 15:39:50,Гагарин,0,0,1,0,0,1,0,...,10,3,1.000000,0,1,32.000000,2.70,4.174387,15.661758,11.502885
17505,6399400,2025-10-16 15:39:50,Гагарин,0,0,1,0,0,1,0,...,10,3,0.666667,0,0,32.000000,2.70,4.174387,15.671715,11.512842
17506,6252400,2025-10-16 15:39:56,Гагарин,0,0,1,0,0,1,0,...,10,3,0.666667,0,0,31.500000,2.70,4.158883,15.648476,11.505351
17507,6316200,2025-10-16 15:39:50,Гагарин,0,0,1,0,0,1,0,...,10,3,0.888889,0,0,31.500000,2.70,4.158883,15.658628,11.515504


In [3]:
unused_features = [
    'Площадь', 'Дата публикации',
    'Цена', 'Цена_log',
    'Цена_за_квадратный_метр', 'Цена_за_квадратный_метр_log'
]

features = data.drop(columns=unused_features)
target = data['Цена_за_квадратный_метр_log']

In [4]:
encoded_data = pd.get_dummies(
    features,
    columns=['Город', 'Субъект РФ'],
    drop_first=True
)

X_tr, X_te, y_tr, y_te = train_test_split(
    encoded_data,
    target,
    test_size=0.2,
    random_state=42
)

In [5]:
norm = StandardScaler()
X_tr_norm = norm.fit_transform(X_tr)
X_te_norm = norm.transform(X_te)

# Линейная регрессия

In [6]:
lin_model = LinearRegression()
lin_model.fit(X_tr_norm, y_tr)

LinearRegression()

In [7]:
log_pred = lin_model.predict(X_te_norm)
log_pred

array([11.93156084, 11.56242022, 11.60636553, ..., 11.66202959,
       11.72355303, 11.45939287])

In [8]:
predicted_price = np.expm1(log_pred)
real_price = np.expm1(y_te)
print(predicted_price)
print(f"\n")
print(real_price)

[151987.60709811 105073.00797522 109793.47974172 ... 116078.3854926
 123444.25176533  94786.50492498]


13313    168181.818182
17485    104545.454545
8753     120238.095238
11201     97895.384615
10723    190362.068966
             ...      
9986      84224.000000
10475    100000.000000
11089    110314.285714
12390    121621.621622
6832      89361.702128
Name: Цена_за_квадратный_метр_log, Length: 3502, dtype: float64


In [9]:
mae_val = mean_absolute_error(real_price, predicted_price)
mape_val = mean_absolute_percentage_error(real_price, predicted_price)
r2_val = r2_score(real_price, predicted_price)

print(f"MAE: {mae_val:.0f} руб/м²")
print(f"MAPE: {mape_val*100:.2f}%")
print(f"R2: {r2_val:.4f}")

MAE: 14673 руб/м²
MAPE: 10.79%
R2: 0.7550


# CatBoost Regressor

In [10]:
cat_model = CatBoostRegressor(
    iterations=800,
    depth=7,
    learning_rate=0.05,
    loss_function='RMSE',
    verbose=False
)

cat_model.fit(X_tr, y_tr)

In [11]:
preds_cat = cat_model.predict(X_te)
preds_cat

array([11.97314783, 11.53069516, 11.67643752, ..., 11.64072571,
       11.66482259, 11.43374289])

In [12]:
predicted_price = np.expm1(preds_cat)
real_price = np.expm1(y_te)
print(predicted_price)
print(f"\n")
print(real_price)

[158441.62676299 101791.85201836 117762.95563908 ... 113631.59960246
 116403.04782928  92386.12290404]


13313    168181.818182
17485    104545.454545
8753     120238.095238
11201     97895.384615
10723    190362.068966
             ...      
9986      84224.000000
10475    100000.000000
11089    110314.285714
12390    121621.621622
6832      89361.702128
Name: Цена_за_квадратный_метр_log, Length: 3502, dtype: float64


In [13]:
mae_val = mean_absolute_error(real_price, predicted_price)
mape_val = mean_absolute_percentage_error(real_price, predicted_price)
r2_val = r2_score(real_price, predicted_price)

print(f"MAE: {mae_val:.0f} руб/м²")
print(f"MAPE: {mape_val*100:.2f}%")
print(f"R2: {r2_val:.4f}")

MAE: 7112 руб/м²
MAPE: 5.25%
R2: 0.9259


# LightGBM

In [14]:
def clean_columns_unique(df):
    new_cols = []
    seen = set()

    for col in df.columns:
        clean = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode()
        clean = re.sub(r"[^A-Za-z0-9_]+", "_", clean)

        if clean == "":
            clean = "col"

        original = clean
        counter = 1
        while clean in seen:
            clean = f"{original}_{counter}"
            counter += 1

        seen.add(clean)
        new_cols.append(clean)

    df = df.copy()
    df.columns = new_cols
    return df

X_tr = clean_columns_unique(X_tr)
X_te = clean_columns_unique(X_te)

In [15]:
lgbm = LGBMRegressor(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=-1,
    num_leaves=50,
)

lgbm.fit(X_tr, y_tr)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007057 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2604
[LightGBM] [Info] Number of data points in the train set: 14007, number of used features: 193
[LightGBM] [Info] Start training from score 11.761378


LGBMRegressor(learning_rate=0.03, n_estimators=600, num_leaves=50)

In [16]:
preds_lgbm = lgbm.predict(X_te)
preds_lgbm

array([11.98697435, 11.53512972, 11.68874572, ..., 11.63009833,
       11.68678449, 11.43664475])

In [17]:
predicted_price = np.expm1(preds_lgbm)
real_price = np.expm1(y_te)
print(predicted_price)
print(f"\n")
print(real_price)

[160647.55189283 102244.26075948 119221.3744351  ... 112430.37663027
 118987.78104079  92654.60701137]


13313    168181.818182
17485    104545.454545
8753     120238.095238
11201     97895.384615
10723    190362.068966
             ...      
9986      84224.000000
10475    100000.000000
11089    110314.285714
12390    121621.621622
6832      89361.702128
Name: Цена_за_квадратный_метр_log, Length: 3502, dtype: float64


In [18]:
mae_val = mean_absolute_error(real_price, predicted_price)
mape_val = mean_absolute_percentage_error(real_price, predicted_price)
r2_val = r2_score(real_price, predicted_price)

print(f"MAE: {mae_val:.0f} руб/м²")
print(f"MAPE: {mape_val*100:.2f}%")
print(f"R2: {r2_val:.4f}")

MAE: 6219 руб/м²
MAPE: 4.58%
R2: 0.9315


# Выводы

# Сравнение моделей: Linear Regression, CatBoost и LightGBM

## 1. Линейная регрессия

### Метрики:
- **MAE:** 14 673 руб/м²  
- **MAPE:** 10.79%  
- **R²:** 0.755  

Линейная модель показала **наиболее слабый результат**.

### Причины:
- Линейная регрессия плохо работает с **нелинейными зависимостями**.  
- Большое число **категориальных признаков** приводит к тому, что модель не захватывает сложные взаимодействия.  
- Логарифмирование целевой переменной помогло стабилизировать обучение, но **гибкости модели всё равно недостаточно** для задач ценообразования в недвижимости.

### Вывод:
Линейная модель подходит только как **базовый бенчмарк**, но не как основная модель.

---

## 2. CatBoost Regressor

### Метрики:
- **MAE:** 7 112 руб/м²  
- **MAPE:** 5.25%  
- **R²:** 0.926  

CatBoost улучшил качество почти **в 2 раза** по сравнению с линейной регрессией.

### Преимущества:
- Эффективно работает с **категориальными признаками** (без необходимости OHE).  
- Хорошо улавливает **нелинейные зависимости**.  
- Обеспечивает стабильное обучение без сложной предобработки.

### Вывод:
CatBoost продемонстрировал **высокое качество**, низкую ошибку и хорошую интерпретируемость.

---

## 3. LightGBM

### Метрики:
- **MAE:** 6 219 руб/м² — **лучший результат**  
- **MAPE:** 4.58% — **лучший результат**  
- **R²:** 0.9315 — **лучший результат**  

LightGBM превзошёл обе другие модели.

### Преимущества:
- Быстрое обучение.  
- Глубокие деревья улавливают сложные паттерны в данных.  
- Прекрасно масштабируется на большое число признаков.  
- Хорошо работает с большим количеством числовых и OHE-признаков.

### Особенность:
- Модель требует **строгих правил к именам признаков**, что вызвало ошибки — проблема была решена очисткой и дедупликацией столбцов.

### Вывод:
**LightGBM — лучшая модель среди протестированных**, обеспечившая минимальную ошибку и максимальную точность.

---


# Итоговое сравнение моделей

| Модель                | MAE (руб/м²) | MAPE      | R²         |
| --------------------- | ------------ | --------- | ---------- |
| **Linear Regression** | 14 673       | 10.79%    | 0.755      |
| **CatBoost**          | 7 112        | 5.25%     | 0.926      |
| **LightGBM**          | **6 219**    | **4.58%** | **0.9315** |
